# Make poster plots — IQ-Learn vs CSIL vs CSIL+SOAR

Generates publication-quality plots from the CSV logs in `logs/`.

**Output:**
- 6 learning-curve plots (3 algos × 2 envs)
- 2 sample-efficiency plots (one per env, all 3 algos overlaid)
- 1 combined summary figure

All plots saved to `plots/` on Drive at 300 dpi.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/imitation_learning

In [ ]:
import os, csv, glob
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)

# Style — make plots look clean
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2.0,
})

ALGOS = ['iqlearn', 'csil', 'csilsoar']
ALGO_LABEL = {'iqlearn': 'IQ-Learn', 'csil': 'CSIL', 'csilsoar': 'CSIL+SOAR'}
ALGO_COLOR = {'iqlearn': '#1f77b4', 'csil': '#ff7f0e', 'csilsoar': '#2ca02c'}
ENVS = ['CartPole', 'Pendulum']
K_VALUES = [1, 3, 5, 10, 15]
SEEDS = [42, 43, 44]

## 2. Load all CSVs into a nested dict

Handles both naming conventions:
- IQ-Learn: `iqlearn_CartPole-v1_K{K}_seed{seed}.csv`
- CSIL/CSILSOAR: `csil_CartPole_K{K}_seed{seed}.csv`

In [ ]:
def load_one(path):
    """Read a CSV and return (steps, rewards) as numpy arrays."""
    steps, rewards = [], []
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            steps.append(int(row['step']))
            rewards.append(float(row['eval_reward']))
    return np.array(steps), np.array(rewards)

def candidate_paths(algo, env, K, seed):
    suffixes = [env, f'{env}-v1']  # try both naming styles
    return [f'logs/{algo}_{s}_K{K}_seed{seed}.csv' for s in suffixes]

# data[algo][env][K][seed] = (steps, rewards)
data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
for algo in ALGOS:
    for env in ENVS:
        for K in K_VALUES:
            for seed in SEEDS:
                for p in candidate_paths(algo, env, K, seed):
                    if os.path.exists(p):
                        data[algo][env][K][seed] = load_one(p)
                        break

# Inventory check
print('Loaded CSVs:')
for algo in ALGOS:
    for env in ENVS:
        n = sum(len(data[algo][env][K]) for K in K_VALUES)
        print(f'  {algo:10s} {env:10s}: {n}/15')

## 3. Aggregation helper

Uses **median + IQR** instead of mean ± std. Median is robust to single-seed failures (which dragged down our earlier mean plots).

In [ ]:
def aggregate(seed_dict):
    """Take {seed: (steps, rewards)} and return (steps, median, low, high)
    truncated to min length so plots line up across seeds with different
    total_steps."""
    if not seed_dict:
        return None
    L = min(len(v[0]) for v in seed_dict.values())
    steps = list(seed_dict.values())[0][0][:L]
    rewards = np.array([v[1][:L] for v in seed_dict.values()])
    median = np.median(rewards, axis=0)
    p25 = np.percentile(rewards, 25, axis=0)
    p75 = np.percentile(rewards, 75, axis=0)
    return steps, median, p25, p75

## 4. Learning curves — 6 plots (3 algos × 2 envs)

In [ ]:
K_COLORS = plt.cm.viridis(np.linspace(0.15, 0.85, len(K_VALUES)))

def plot_learning_curves(algo, env, ax=None, save=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4.5))
    for K, color in zip(K_VALUES, K_COLORS):
        agg = aggregate(data[algo][env][K])
        if agg is None:
            continue
        steps, median, p25, p75 = agg
        ax.plot(steps, median, color=color, label=f'K={K}', linewidth=2)
        ax.fill_between(steps, p25, p75, color=color, alpha=0.18)
    ax.set_xlabel('Training steps')
    ax.set_ylabel('Evaluation reward (median over 3 seeds)')
    ax.set_title(f'{ALGO_LABEL[algo]} — {env}-v1')
    ax.legend(loc='best', frameon=True, framealpha=0.9)
    if save:
        path = f'plots/learning_curves_{algo}_{env}.png'
        plt.gcf().savefig(path)
        print(f'  saved {path}')

for algo in ALGOS:
    for env in ENVS:
        if not data[algo][env]:
            print(f'  SKIP {algo} {env} (no data)')
            continue
        plt.figure()
        plot_learning_curves(algo, env)
        plt.show()

## 5. Sample efficiency — 2 plots, all 3 algorithms overlaid

**This is the headline figure** — it shows the comparison the project asks for.

In [ ]:
def max_reward_per_seed(seed_dict):
    """Best reward each seed reached during its run."""
    return [v[1].max() for v in seed_dict.values()]

def plot_sample_efficiency(env, ax=None, save=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4.5))
    for algo in ALGOS:
        xs, medians, p25s, p75s = [], [], [], []
        for K in K_VALUES:
            maxes = max_reward_per_seed(data[algo][env][K])
            if not maxes:
                continue
            xs.append(K)
            medians.append(np.median(maxes))
            p25s.append(np.percentile(maxes, 25))
            p75s.append(np.percentile(maxes, 75))
        if not xs:
            continue
        medians, p25s, p75s = map(np.array, [medians, p25s, p75s])
        ax.errorbar(xs, medians,
                    yerr=[medians - p25s, p75s - medians],
                    color=ALGO_COLOR[algo], label=ALGO_LABEL[algo],
                    marker='o', capsize=4, linewidth=2)
    ax.set_xlabel('K (number of expert trajectories)')
    ax.set_ylabel('Best evaluation reward (median over 3 seeds)')
    ax.set_title(f'Sample efficiency — {env}-v1')
    ax.set_xscale('log')
    ax.set_xticks(K_VALUES)
    ax.set_xticklabels(K_VALUES)
    ax.legend(loc='best', frameon=True, framealpha=0.9)
    if save:
        path = f'plots/sample_efficiency_{env}.png'
        plt.gcf().savefig(path)
        print(f'  saved {path}')

for env in ENVS:
    plt.figure()
    plot_sample_efficiency(env)
    plt.show()

## 6. Combined comparison figure (poster ready)

All 8 panels in one figure: 6 learning curves + 2 sample efficiency. Good for the poster's main visual block.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 18))

# Top 3 rows: learning curves
for row, algo in enumerate(ALGOS):
    for col, env in enumerate(ENVS):
        if not data[algo][env]:
            axes[row][col].axis('off')
            continue
        plot_learning_curves(algo, env, ax=axes[row][col], save=False)

# Bottom row: sample efficiency overlay
for col, env in enumerate(ENVS):
    plot_sample_efficiency(env, ax=axes[3][col], save=False)

plt.tight_layout()
fig.savefig('plots/combined_all.png')
print('  saved plots/combined_all.png')
plt.show()

## 7. Quick numerical summary table

In [ ]:
print(f'\n{"Algorithm":<12} {"Env":<10} {"K":<4} {"max (median)":>14} {"max (best)":>12} {"n_seeds":>9}')
print('-' * 65)
for algo in ALGOS:
    for env in ENVS:
        for K in K_VALUES:
            maxes = max_reward_per_seed(data[algo][env][K])
            if not maxes:
                continue
            print(f'{ALGO_LABEL[algo]:<12} {env:<10} {K:<4} '
                  f'{np.median(maxes):>14.1f} {max(maxes):>12.1f} {len(maxes):>9}')
        print()

## 8. (Optional) Download all plots as a zip

In [ ]:
!cd plots && zip -r ../plots.zip . && cd ..
from google.colab import files
files.download('plots.zip')